# NeuralEvolve: evolving fast networks for the Apple Neural Engine

NSGA-II searches network **architecture + per-stage compression** for three objectives at once:
**error**, **latency on the Apple Neural Engine (ANE)** and **model size**. Weights are always trained by SGD
(in a weight-sharing supernet); evolution only chooses the blueprint.

This notebook only *reads* results. The heavy work is done by the scripts listed in `README.md`.
Every section prints `missing: ...` until its step has been run.

In [ ]:
%matplotlib inline
import random
import pandas as pd
import torch
import genome as G
import analysis as A
pd.set_option('display.precision', 4)

## 1. The genome

91 integers: 1 resolution gene, 6 stages × (depth, width, bits), 24 block slots × (kernel, expansion, SE).
Block slots beyond a stage's depth are inactive. Below: the smallest, middle and largest network.

In [ ]:
rows = []
for name, g in [('min', G.extreme_genome('min')), ('mid', G.extreme_genome('mid')),
                ('max', G.extreme_genome('max')), ('random', G.random_genome(random.Random(0)))]:
    d = G.describe(g)
    c = G.cost_measures(g)
    rows.append({'genome': name, 'resolution': d['resolution'],
                 'depths': [s['depth'] for s in d['stages']], 'widths': [s['width'] for s in d['stages']],
                 'bits': [s['bits'] for s in d['stages']], 'MParams': c['params'] / 1e6,
                 'MMACs': c['macs'] / 1e6, 'size_MB': c['size_mb']})
pd.DataFrame(rows)

In [ ]:
# genetic operators on two random parents
rng = random.Random(1)
a, b = G.random_genome(rng), G.random_genome(rng)
child = G.mutate(G.crossover_uniform(a, b, rng), rng=rng)
print('parent a:', G.genome_key(a))
print('parent b:', G.genome_key(b))
print('child   :', G.genome_key(child))
model, res, bits = G.decode(child)
print(model(torch.randn(1, 3, res, res)).shape)

## 2. Supernet training
Sandwich rule: largest + smallest + 2 random subnets per batch, in-place distillation from the largest.

In [ ]:
A.supernet_training()

## 3. Is MACs a good proxy for ANE latency?
Real Core ML measurements of ~400 random networks on the Mac's Neural Engine.

In [ ]:
A.macs_vs_latency()

## 4. Surrogate predictors
Held-out quality of the latency MLP and the accuracy MLP (accuracy targets come from the supernet).

In [ ]:
A.predictor_quality()

## RQ1: Is NSGA-II better than simpler search at the same budget?
Hypervolume of all evaluated points (normalized with the same bounds for every algorithm), 5 seeds.
`p_vs_nsga2` is a two-sided Mann-Whitney U test.

In [ ]:
A.rq1()

## RQ2: Which speed objective finds the best networks *in reality*?
Fronts found with MACs, parameter count or the latency predictor, all judged with measured ANE latency.

In [ ]:
A.rq2()

## RQ3: Searching architecture and compression together vs one after the other

In [ ]:
A.rq3()

## RQ4: GA ablations
One change at a time from the default (population 50, uniform crossover, mutation 1/91, repair on).

In [ ]:
A.rq4()

## RQ5: Can the supernet be trusted?
Supernet accuracy vs accuracy of the same networks trained from scratch.

In [ ]:
A.rq5()

## RQ6: Does the front transfer between devices?
Mac CPU / GPU / Neural Engine and iPhone (from the Xcode performance report).

In [ ]:
A.rq6()

## RQ7: What did evolution learn about the hardware?
Gene statistics per stage on the final Pareto front vs everything the search evaluated.

In [ ]:
A.rq7()

## Final result: trained front vs MobileNetV2

In [ ]:
A.final_front()